# 货架装箱问题

**类别：** 装箱

来源： [https://www.hexaly.com/templates/shelf-packing-problem](https://www.hexaly.com/templates/shelf-packing-problem)


## 问题

在 Shelf Packing Problem 中，一组具有已知重量和已知类别的物品必须被分配到位于货架上的、具有统一容量的箱子中。货架上的总重量以及属于给定类别的物品数量均受到限制。目标是最小化用于放置所有物品的货架数量（第一个目标函数），然后最小化箱子数量（第二个目标函数）。该问题是 [Bin Packing Problem (BPP)](https://www.hexaly.com/docs/last/exampletour/binpacking.html) 的一个变种，因此是 NP-hard 的。

	

### 学到的建模原则

- 使用 [JSON](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/jsonmodule.html) 文件
- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模箱子的内容
- 使用 [union operator](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#union) 来建模货架的内容
- 定义 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算箱子和货架的总重量


## 数据

Shelf Packing Problem 实例是随机生成的 JSON 文件。它们由以下元素组成：

- “nbItems”
- “nbShelves”
- “nbCategories”
- “binCapacity”
- “shelfCapacity”
- “itemWeights”
- “itemCategories”
- “limitCategories”：limitCategories[c] 是货架 s 上类别 c 的最大物品数量
- “binsAssignment”：binsAssignment[s] 是分配给货架 s 的箱子列表

我们为每种 API 模型使用 JSON 库来读取实例数据并写出输出解：C#（Newtonsoft.Json）、Java（gson-2.8.8）、Python（json）、C++（nlohmann/json.hpp）。对于 Hexaly Modeler，使用 [JSON module](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/jsonmodule.html)。


## 程序

此处实现的模型使用了 [set](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#modeling-with-sets) 变量。

每个货架上有若干箱子，对每个货架，我们定义一个中间表达式作为分配到该特定货架的箱子中物品的 ‘[union](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#union)‘。

对于每个箱子，我们定义一个 set 来描述分配到该箱子中的物品。这些 set 被约束形成 ‘[partition](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition)‘，这意味着每个物品必须被分配到恰好一个箱子。
分配到一个箱子中的物品的重量之和不得超过箱子容量。
货架上所有物品的重量之和不得超过货架容量。
在任何货架上，给定类别的物品数量不得超过设定的限制。为了实现该约束，我们对货架上的物品与该类别的物品执行 ‘[intersection](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#intersection)‘，并将其计数限制在 limitCategories[c] 阈值以下。

目标是首先最小化使用的货架数量（第一个目标），然后最小化使用的箱子数量（第二个目标）。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import json
import sys
import math

if len(sys.argv) < 2:
    print("Usage: python shelf_packing.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)

with hexaly.optimizer.HexalyOptimizer() as optimizer:
    # Read instance data
    filename = sys.argv[1]
    with open(filename, 'r') as file:
        data = json.load(file)
        nb_items = data["nbItems"]
        nb_categories = data["nbCategories"]
        nb_shelves = data["nbShelves"]
        bin_capacity = data["binCapacity"]
        shelf_capacity = data["shelfCapacity"]
        weights_data = data["itemWeights"]
        categories_data = data["itemCategories"]

        # limitCategories[c]: maximum number of items of category c on a shelf
        limit_categories = data["limitCategories"]
        # binAssignment[s]: list of bins assigned to shelf s               
        bins_assignment = data["binsAssignment"]

    nb_max_bins = nb_items

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # Set decisions: bins[k] represents the items in bin k
    bins = [model.set(nb_items) for _ in range(nb_max_bins)]

    # Shelves are the sets of items contained in the union of bins belonging to said shelf
    shelves = [model.union(bins[b] for b in bins_assignment[s]) for s in range(nb_shelves)]

    # Constraint: Each item must be in one bin and one bin only
    model.constraint(model.partition(bins))

    # Create an array and a function to retrieve the item's weight
    weights = model.array(weights_data)
    categories = model.array(categories_data)
    weight_lambda = model.lambda_function(lambda i: weights[i])

    # Constraint: Weight constraint on each bin
    bin_weights = [model.sum(b, weight_lambda) for b in bins]
    for w in bin_weights:
        model.constraint(w <= bin_capacity)

    # Bin b is used if at least one item is in it
    bins_used = [model.count(b) > 0 for b in bins]

    # Constraint: Weight constraint on each shelf
    shelves_weights = [model.sum(s, weight_lambda) for s in shelves]
    for w in shelves_weights:
        model.constraint(w <= shelf_capacity)

    # Constraint: the number of items per category on a shelf is limited
    for c in range(nb_categories):
        items_per_category = []
        for i in range(nb_items):
            if categories_data[i] == c:
                items_per_category.append(i)
        items_per_category_array = model.array(items_per_category)
        for s in shelves:
            model.constraint(model.count(model.intersection(s, items_per_category_array)) <= limit_categories[c])

    # Shelf s is used if at least one item is on it
    shelves_used = [model.count(s) > 0 for s in shelves]

    total_shelves_used = model.sum(shelves_used)
    total_bins_used = model.sum(bins_used)

    # Minimize the number of used shelves (first objective) then bins (second objective)
    model.minimize(total_shelves_used)
    model.minimize(total_bins_used)
    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    optimizer.solve()

    # Write the solution in a file
    if len(sys.argv) >= 3:
        with open(sys.argv[2], 'w') as f:
            for s in range(nb_shelves):
                if shelves_used[s].value:
                    f.write("Shelf %d total weight: %d/%d \n" %(s, shelves_weights[s].value, shelf_capacity) )
                    for k in range(nb_max_bins):
                        if bins_used[k].value:
                            f.write(">Bin weight: %d/%d | Items: " % (bin_weights[k].value, bin_capacity))
                            for e in bins[k].value:
                                f.write("#%d " % e)
                            f.write("\n")
